In [ ]:
%pip install pandas


In [ ]:


import pandas as pd
from rdkit import Chem

# Load your own CSV
df = pd.read_csv("LOTUS.csv")
df


In [ ]:
# Checking for missing values in the dataset
df.isna().sum()

💊 **Lipinski’s Rule of Five**

Lipinski’s Rule of Five is a widely used guideline in drug discovery to evaluate the oral bioavailability of a compound. It helps identify molecules that are more likely to become orally active drugs in humans.

A compound is considered likely to have good oral bioavailability if **no more than one** of the following rules is violated:

- **Molecular weight** ≤ 500 Da  
- **LogP** (octanol–water partition coefficient) ≤ 5  
- **Hydrogen bond donors** (OH and NH groups) ≤ 5  
- **Hydrogen bond acceptors** (N and O atoms) ≤ 10  


In [ ]:
# Converting SMILES strings into RDKit molecule objects
df['Mol'] = df['SMILES'].apply(Chem.MolFromSmiles)

# Checking for missing values (invalid SMILES will result in a 'None' or NaN in the 'Mol' column)
df.isna().sum()

# Removing rows where molecule conversion failed
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)
df

💊 **Lipinski’s Rule of Five**

Lipinski’s Rule of Five is a widely used guideline in drug discovery to evaluate the oral bioavailability of a compound. It helps identify molecules that are more likely to become orally active drugs in humans.

A compound is considered likely to have good oral bioavailability if **no more than one** of the following rules is violated:

- **Molecular weight** ≤ 500 Da  
- **LogP** (octanol–water partition coefficient) ≤ 5  
- **Hydrogen bond donors** (OH and NH groups) ≤ 5  
- **Hydrogen bond acceptors** (N and O atoms) ≤ 10  


In [ ]:
from rdkit.Chem import Descriptors
from rdkit.Chem.Descriptors import rdMolDescriptors

# Calculating Molecular Weight for each molecule
df['MolWt'] = df['Mol'].apply(Descriptors.MolWt)
df

In [ ]:
# Calculating LogP (Partition Coefficient)
df['LogP'] = df['Mol'].apply(Descriptors.MolLogP)

# Counting Hydrogen Bond Donors
df['HbondDonors'] = df['Mol'].apply(Descriptors.NumHDonors)

# Counting Hydrogen Bond Acceptors
df['HbondAcceptors'] = df['Mol'].apply(Descriptors.NumHAcceptors)

# View the updated dataframe with all descriptor columns
df

In [ ]:
# Filtering the dataframe based on Lipinski's Rule of Five
df_filtered = df[
    (df['MolWt'] <= 500) & 
    (df['LogP'] <= 5) & 
    (df['HbondDonors'] <= 5) & 
    (df['HbondAcceptors'] <= 10)
].copy()

# Resetting the index for the new filtered dataset
df_filtered.reset_index(inplace=True, drop=True)

# View the final prioritized compounds
df_filtered

In [ ]:
# Total compounds after cleaning
total_compounds = len(df)

# Compounds that passed Lipinski
passed = len(df_filtered)

# Compounds that failed Lipinski
failed = total_compounds - passed

print("Total:", total_compounds)
print("Passed:", passed)
print("Failed:", failed)


In [ ]:
import matplotlib.pyplot as plt

# Calculate the statistics
total_compounds = len(df)
passed = len(df_filtered)
failed = total_compounds - passed

labels = ['Passed Lipinski', 'Failed Lipinski']
values = [passed, failed]

# Create the visualization
plt.figure(figsize=(8, 6))
# Using professional colors: Dark blue-gray for passed, soft red for failed
bars = plt.bar(labels, values, width=0.6, color=['#2C3E50', '#A93226'])

# Adjust Y-axis limit for labels
max_value = max(values)
plt.ylim(0, max_value * 1.25)

# Add count and percentage labels above bars
for i, v in enumerate(values):
    percentage = (v / total_compounds) * 100
    plt.text(i, v + (max_value * 0.03), f"{v:,}\n({percentage:.1f}%)", 
             ha='center', fontsize=11, fontweight='bold')

plt.title("Lipinski Rule of Five Screening Results", fontsize=14, fontweight='bold')
plt.ylabel("Number of Compounds", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure seaborn style is clean
sns.set_style("whitegrid")

# Set up a 2x2 grid for the plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reduce title padding and set bold title
fig.suptitle('Physicochemical Profile of LOTUS Library', fontsize=18, fontweight='bold', y=0.98)

# 1. Molecular Weight
sns.histplot(df['MolWt'], bins=50, ax=axes[0, 0], color='#2C3E50', kde=True, alpha=0.8)
axes[0, 0].axvline(500, color='#A93226', linestyle='--', linewidth=2, label='Limit (500)')
axes[0, 0].set_title('Molecular Weight (Da)', fontweight='bold')
axes[0, 0].legend()

# 2. LogP
sns.histplot(df['LogP'], bins=50, ax=axes[0, 1], color='#2C3E50', kde=True, alpha=0.8)
axes[0, 1].axvline(5, color='#A93226', linestyle='--', linewidth=2, label='Limit (5)')
axes[0, 1].set_title('LogP (Octanol/Water Partition)', fontweight='bold')
axes[0, 1].legend()

# 3. H-Bond Donors
# Using discrete bins since these are integers
sns.histplot(df['HbondDonors'], bins=range(0, 15), ax=axes[1, 0], color='#2C3E50', alpha=0.8)
axes[1, 0].axvline(5, color='#A93226', linestyle='--', linewidth=2, label='Limit (5)')
axes[1, 0].set_title('H-Bond Donors (HBD)', fontweight='bold')
axes[1, 0].legend()

# 4. H-Bond Acceptors
sns.histplot(df['HbondAcceptors'], bins=range(0, 20), ax=axes[1, 1], color='#2C3E50', alpha=0.8)
axes[1, 1].axvline(10, color='#A93226', linestyle='--', linewidth=2, label='Limit (10)')
axes[1, 1].set_title('H-Bond Acceptors (HBA)', fontweight='bold')
axes[1, 1].legend()

# Adjust layout: rect=[left, bottom, right, top]
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Save for your Word report
plt.savefig("LOTUS_Physicochemical_Dist.png", dpi=300, bbox_inches='tight')
plt.show()

⚖️ **Quantitative Estimate of Drug-Likeness (QED)**

QED is a metric that quantifies how "drug-like" a molecule is by combining several important physicochemical properties into a single score. It outputs a value between **0 and 1**, where higher scores indicate better drug-likeness.

**Properties Considered in QED:**

- Molecular weight (MW)  
- LogP (lipophilicity)  
- Hydrogen bond donors (HBD) and acceptors (HBA)  
- Rotatable bonds  
- Aromatic ring count  
- Structural alerts  
- Topological polar surface area (TPSA)  

**QED Score Interpretation:**

| QED Score | Interpretation |
|-----------|----------------|
| 0.9 – 1.0 | Highly drug-like |
| 0.7 – 0.9 | Moderately drug-like |
| 0.4 – 0.7 | Potentially drug-like |
| < 0.4     | Poor drug-likeness |

> **Note:** Most marketed oral drugs tend to have QED scores above 0.67, though this is a guideline rather than a strict cutoff. QED is especially useful for **ranking and prioritizing compounds** rather than just binary filtering.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculation (Make sure this runs first)
from rdkit.Chem import QED
df_filtered['Qed'] = df_filtered['Mol'].apply(QED.qed)

# 2. Plotting
plt.figure(figsize=(10, 6))

# Histogram with Kernel Density Estimate
sns.histplot(df_filtered['Qed'], bins=40, kde=True, color='#2C3E50', alpha=0.8)

# Add the "Drug-likeness" threshold line
plt.axvline(0.8, color='#A93226', linestyle='--', linewidth=2, label='High Quality Gate (0.8)')

# Fill the "High Quality" area to make it stand out
plt.axvspan(0.8, 1.0, alpha=0.1, color='#27AE60', label='Drug-like Zone')

# 3. Formatting to match your previous style
plt.title('Quantitative Estimate of Drug-likeness (QED) Distribution', 
          fontsize=14, fontweight='bold', pad=10)
plt.xlabel('QED Score', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(0, 1) # QED is always between 0 and 1
plt.legend(frameon=True, loc='upper left')

plt.tight_layout()
plt.savefig("LOTUS_QED_Distribution.png", dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics for your report
print(f"Mean QED: {df_filtered['Qed'].mean():.3f}")
print(f"Compounds with QED >= 0.8: {len(df_filtered[df_filtered['Qed'] >= 0.8])}")

In [ ]:
# Filtering for highly drug-like compounds (QED >= 0.67) from the Lipinski-filtered set
df_filtered_qed = df_filtered[df_filtered['Qed'] >= 0.8].copy()

# Reset index for cleanliness
df_filtered_qed.reset_index(inplace=True, drop=True)

# View the highly drug-like compounds
df_filtered_qed


🧪 **Synthetic Accessibility (SA) Score**

The Synthetic Accessibility (SA) Score estimates how easy or difficult it would be to chemically synthesize a compound in the laboratory. It is a vital metric in drug discovery for prioritizing molecules that are not only active but also practical to manufacture.

**How It Works:**

The SA score combines two main components:

- **Fragment contributions** — How common or rare molecular fragments are based on a large dataset of known synthetic compounds.  
- **Complexity penalties** — Structural features that increase synthetic difficulty, such as:
  - Large ring systems  
  - Highly branched structures  
  - Chiral centers  
  - Unusual functional groups  

**SA Score Range and Interpretation:**

| SA Score | Interpretation |
|----------|----------------|
| 1.0 – 3.0 | Easy to synthesize |
| 3.1 – 5.0 | Moderate difficulty |
| 5.1 – 10.0 | Difficult to synthesize |

> **Note:** Lower SA scores are more desirable, indicating a higher likelihood that a compound can be successfully synthesized in the lab. Using the SA score helps avoid “dead-end” compounds early in the drug discovery pipeline.


In [ ]:
import urllib.request
import os

# URLs for the SA Scorer components
sascorer_url = "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py"
fpscores_url = "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz"

# Download files if they don't exist
if not os.path.exists("sascorer.py"):
    urllib.request.urlretrieve(sascorer_url, "sascorer.py")
    print("Downloaded sascorer.py ✅")

if not os.path.exists("fpscores.pkl.gz"):
    urllib.request.urlretrieve(fpscores_url, "fpscores.pkl.gz")
    print("Downloaded fpscores.pkl.gz ✅")

print("SA Scorer setup complete.")

In [ ]:
import sascorer
print(sascorer.__file__)


In [ ]:
import sascorer
import seaborn as sns
import matplotlib.pyplot as plt

# 1. THE MISSING STEP: Run the calculation to create the 'sascore' column
# This might take a few seconds depending on the size of your dataset
print("Calculating SA Scores... please wait.")
df_filtered_qed['sascore'] = df_filtered_qed['Mol'].apply(sascorer.calculateScore)
print("Calculation complete! Generating plot...")

# 2. Setup the Plot
plt.figure(figsize=(10, 6))

# Histogram with KDE
sns.histplot(df_filtered_qed['sascore'], bins=30, kde=True, color='#2C3E50', alpha=0.7)

# Add the "Synthetically Feasible" threshold line (3.0)
plt.axvline(3.0, color='#A93226', linestyle='--', linewidth=2, label='Feasibility Gate (3.0)')

# Fill the "Easy to Synthesize" area (1.0 to 3.0)
plt.axvspan(1.0, 3.0, alpha=0.15, color='#27AE60', label='High Feasibility Zone')

# 3. Professional Formatting
plt.title('Synthetic Accessibility (SA) Score Distribution', fontsize=14, fontweight='bold', pad=10)
plt.xlabel('SA Score (1 = Easy, 10 = Difficult)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(1, 10) 
plt.legend(frameon=True, loc='upper right')

plt.tight_layout()
plt.show()

# 4. Summary for your report text
easy_to_synth = len(df_filtered_qed[df_filtered_qed['sascore'] <= 3.0])
print(f"Mean SA Score: {df_filtered_qed['sascore'].mean():.2f}")
print(f"Final Count (SA <= 3.0): {easy_to_synth:,} compounds")

In [ ]:
# Filtering QED-passed molecules with acceptable synthetic accessibility (SA ≤ 3)
df_filtered_sascore = df_filtered_qed[df_filtered_qed['sascore'] <= 3.0].copy()

# Reset index for cleanliness
df_filtered_sascore.reset_index(drop=True, inplace=True)

# Display the final filtered compounds
df_filtered_sascore


In [ ]:
from rdkit import Chem
from rdkit.Chem import FilterCatalog


In [ ]:
# Build PAINS catalog (Baell & Holloway, JMC 2010)
params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(params)


In [ ]:
pains_flags = []
pains_hits = []

for smi in df_filtered_sascore['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    
    if mol is None:
        pains_flags.append(None)
        pains_hits.append(None)
    else:
        matches = pains_catalog.GetMatches(mol)
        pains_flags.append(len(matches) > 0)
        pains_hits.append("; ".join([m.GetDescription() for m in matches]))


In [ ]:
df_filtered_sascore['PAINS_flag'] = pains_flags
df_filtered_sascore['PAINS_alerts'] = pains_hits


In [ ]:
df_pains_free = df_filtered_sascore[df_filtered_sascore['PAINS_flag'] == False].copy()
df_pains_free.reset_index(drop=True, inplace=True)


In [ ]:
df_pains_flagged = df_filtered_sascore[df_filtered_sascore['PAINS_flag'] == True].copy()
df_pains_flagged.reset_index(drop=True, inplace=True)


In [ ]:
df_filtered_sascore[['SMILES', 'PAINS_flag', 'PAINS_alerts']].head(10)


In [ ]:
df_filtered_sascore['PAINS_flag'].value_counts()


In [ ]:
df_pains_only = df_filtered_sascore[
    df_filtered_sascore['PAINS_flag'] == True
]

pains_counts = (
    df_pains_only['PAINS_alerts']
    .str.split('; ')
    .explode()
    .value_counts()
)

pains_counts.head(10)


In [ ]:
df_pains_only[
    ['SMILES', 'PAINS_alerts']
].head(10)


In [ ]:
import matplotlib.pyplot as plt

pains_counts.head(10).plot(kind='bar')
plt.ylabel("Frequency")
plt.title("Top 10 PAINS Substructures")
plt.show()


In [ ]:
# Filter only PAINS-free compounds:
df_hits = df_filtered_sascore[df_filtered_sascore['PAINS_flag'] == False].copy()
df_hits.reset_index(drop=True, inplace=True)


In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import matplotlib.pyplot as plt
import numpy as np  # Added for the data type fix

# 1. Grab the first 10 compounds from your df_hits
top_10_clean = df_hits.head(10)

mols = []
legends = []

for idx, row in top_10_clean.iterrows():
    mol = Chem.MolFromSmiles(row['SMILES'])
    if mol is not None:
        mols.append(mol)
        legends.append(
            f"{row['LOTUS ID']}\nQED: {row['Qed']:.2f} | SA: {row['sascore']:.2f}"
        )

# 2. Generate the Grid Image (explicitly as a PIL image)
img = Draw.MolsToGridImage(
    mols,
    molsPerRow=5,
    subImgSize=(300, 300),
    legends=legends,
    returnPNG=False # This ensures a PIL object is returned
)

# 3. Add the Title and Display (Using NumPy conversion to fix the TypeError)
plt.figure(figsize=(15, 8))
plt.imshow(np.asarray(img)) # The FIX: convert 'object' to 'numpy array'
plt.axis('off')

# Compact Title
plt.title("Initial 10 PAINS-Free Lead Candidates", 
          fontsize=16, fontweight='bold', pad=10, color='#2C3E50')

plt.tight_layout()
plt.savefig("PAINS_Free_Hits.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
import textwrap

# 1. Grab the first 10 molecules UNRANKED (Exactly as they appear in df_hits)
df_top_10 = df_hits.head(10).reset_index(drop=True)

# 2. Prepare the molecules
mols = [row['Mol'] for _, row in df_top_10.iterrows() if row['Mol'] is not None]

# 3. Generate the high-resolution grid WITHOUT legends
grid_hits = Draw.MolsToGridImage(
    mols,
    molsPerRow=5,
    subImgSize=(450, 450),
    legends=["" for _ in mols], 
    returnPNG=False
)

# 4. Setup White Header Space
width, height = grid_hits.size
header_h = 200
final_img = Image.new("RGB", (width, height + header_h), "white")
final_img.paste(grid_hits, (0, header_h))

draw = ImageDraw.Draw(final_img)

# 5. Define Fonts
try:
    font_title = ImageFont.truetype("arialbd.ttf", 50)
    font_label = ImageFont.truetype("arialbd.ttf", 45)
except:
    font_title = ImageFont.load_default()
    font_label = ImageFont.load_default()

# 6. Add A–J Labels to each grid cell
labels = list("ABCDEFGHIJ")
for i in range(len(mols)):
    row, col = i // 5, i % 5
    x = col * 450 + 20
    # Adjusted y to account for header
    y = row * 450 + header_h + 20
    draw.text((x, y), f"{labels[i]}.", fill="black", font=font_label)

# 7. Add Professional Main Title (Updated to reflect "Initial Hits")
main_title = "INITIAL 10 PAINS-FREE VIRTUAL SCREENING HITS"
wrapped_title = textwrap.fill(main_title, width=60)

bbox = draw.multiline_textbbox((0, 0), wrapped_title, font=font_title, align="center")
t_w, t_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
draw.multiline_text(((width - t_w) // 2, (header_h - t_h) // 2), 
                    wrapped_title, fill="black", font=font_title, align="center")

# 8. Subtle separator line
draw.line([(50, header_h), (width-50, header_h)], fill="#D3D3D3", width=3)

# 9. Final display
display(final_img)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import matplotlib.pyplot as plt
import numpy as np

# 1. Grab the first 10 compounds that HAVE PAINS alerts
top_10_pains = df_pains_only.head(10)

mols = []
legends = []

for idx, row in top_10_pains.iterrows():
    mol = Chem.MolFromSmiles(row['SMILES'])
    if mol is not None:
        mols.append(mol)
        # We display the LOTUS ID and the first PAINS alert found
        # (Shortening the alert string in case it's very long)
        alert = row['PAINS_alerts'].split(';')[0][:25] 
        legends.append(f"{row['LOTUS ID']}\nAlert: {alert}...")

# 2. Generate the Grid Image
img = Draw.MolsToGridImage(
    mols,
    molsPerRow=5,
    subImgSize=(300, 300),
    legends=legends,
    returnPNG=False
)

# 3. Display with the NumPy fix to prevent TypeErrors
plt.figure(figsize=(15, 8))
plt.imshow(np.asarray(img))
plt.axis('off')

# Professional Header for the 'Rejected' set
plt.title("Example Compounds Flagged with PAINS Substructures (Excluded)", 
          fontsize=16, fontweight='bold', pad=10, color='#A93226') # Using Red for 'Warning'

plt.tight_layout()
plt.savefig("PAINS_Flagged_Compounds.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import textwrap
from IPython.display import display

# 1. Prepare the molecules (Top 8 for a 2x4 grid)
top_pains = df_pains_only.head(8) 
mols = [Chem.MolFromSmiles(row['SMILES']) for _, row in top_pains.iterrows()]
mols = [m for m in mols if m is not None]

# 2. Generate Grid WITHOUT legends (Keeps the space under molecules clean)
grid_img = Draw.MolsToGridImage(
    mols,
    molsPerRow=4,
    subImgSize=(450, 450),
    legends=["" for _ in mols], 
    returnPNG=False
)

# 3. Setup Header Space
width, height = grid_img.size
header_h = 200 
final_img = Image.new("RGB", (width, height + header_h), "white")
final_img.paste(grid_img, (0, header_h))

draw = ImageDraw.Draw(final_img)

# 4. Define Fonts
try:
    font_title = ImageFont.truetype("arialbd.ttf", 45) 
    font_label = ImageFont.truetype("arialbd.ttf", 50) 
except:
    font_title = ImageFont.load_default()
    font_label = ImageFont.load_default()

# 5. Add A–H Labels inside the grid cells
labels = list("ABCDEFGH")
for i in range(len(mols)):
    row, col = i // 4, i % 4
    x = col * 450 + 20
    y = row * 450 + header_h + 20
    draw.text((x, y), f"{labels[i]}.", fill="black", font=font_label)

# 6. Center the Main Title in the Header
main_title = "PAINS-FLAGGED COMPOUNDS SHOWING INTERFERENCE MOTIFS (A–H)"
wrapped_title = textwrap.fill(main_title, width=50)

bbox = draw.multiline_textbbox((0, 0), wrapped_title, font=font_title, align="center")
t_w, t_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
draw.multiline_text(((width - t_w) // 2, (header_h - t_h) // 2), 
                    wrapped_title, fill="black", font=font_title, align="center")

# 7. Subtle Separator Line
draw.line([(50, header_h), (width-50, header_h)], fill="#D3D3D3", width=3)

# 8. FINAL DISPLAY (Directly in VS Code output)
display(final_img)

In [ ]:
print("Total compounds after SA filter:", len(df_filtered_sascore))
print("PAINS-free compounds:", len(df_hits))
print("PAINS-flagged compounds:", len(df_pains_flagged))


In [ ]:
# Select only required columns
df_export = df_hits_sorted[['LOTUS ID', 'SMILES']].copy()

# Export to CSV
df_export.to_csv("Final_PAINS_Free_ID_SMILES.csv", index=False)

print("CSV exported successfully ✅")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Data Prep
total = total_compounds
p_lipinski = passed
p_qed = len(df_filtered_qed) if 'df_filtered_qed' in locals() else passed_qed
p_sa = len(df_filtered_sascore) if 'df_filtered_sascore' in locals() else passed_sa
p_final = len(df_hits)

labels = ['Initial\nLibrary', 'Lipinski\nPass', 'QED\nPass', 'SA\nPass', 'Final\nHits']
counts = [total, p_lipinski, p_qed, p_sa, p_final]

# 2. Setup Plot
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(12, 7))

# Use Log Scale to handle the massive range differences
ax.set_yscale('log')

# Professional Deep Gradient
colors = sns.color_palette("mako", len(labels))

bars = ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=1.2, width=0.6)

# 3. Add Labels
for i, bar in enumerate(bars):
    yval = bar.get_height()
    
    # Text label on top of bar (Count)
    ax.text(bar.get_x() + bar.get_width()/2, yval * 1.1, 
             f"{int(yval):,}", ha='center', va='bottom', 
             fontsize=11, fontweight='bold', color='#2C3E50')
    
    # Percentage label at the base of the bar
    pct_of_original = (yval / total) * 100
    ax.text(bar.get_x() + bar.get_width()/2, yval * 0.4 if yval > 10 else yval, 
             f"{pct_of_original:.2f}%", ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if yval > 100 else 'black')

# 4. Add Step-wise Attrition (Drop %)
for i in range(len(counts) - 1):
    retention = (counts[i+1] / counts[i]) * 100
    drop = 100 - retention
    
    # Positioning the text between bars
    mid_point = i + 0.5
    ax.text(mid_point, counts[i+1], f"Drop: {drop:.1f}%", 
            ha='center', va='bottom', fontsize=9, 
            color='#A93226', fontweight='bold', fontstyle='italic')

# 5. Final Polish
plt.title("VIRTUAL SCREENING ATTRITION (LOG SCALE)", fontsize=16, fontweight='bold', pad=20)
plt.ylabel("Number of Compounds (Log$_{10}$)", fontsize=12, fontweight='bold')

# Ensure the y-limit accommodates the labels at the top
ax.set_ylim(bottom=1, top=max(counts) * 5)

sns.despine()
plt.tight_layout()
plt.savefig("Screening_Pipeline_LogScale.png", dpi=300)
plt.show()

In [ ]:
from rdkit import Chem

output_file = "LOTUS_Final_Hits_2692.sdf"
count = 0

# Create a writer
writer = Chem.SDWriter(output_file)

print(f"💾 Saving {len(df_hits)} compounds from dataframe to SDF...")

for idx, row in df_hits.iterrows():
    mol = row['Mol']
    if mol is not None:
        # Set the LOTUS ID as the molecule name inside the SDF
        mol.SetProp("_Name", str(row['LOTUS ID']))
        
        # Optionally add other data to the SDF tags
        mol.SetProp("QED", f"{row['Qed']:.3f}")
        mol.SetProp("SA_Score", f"{row['sascore']:.3f}")
        
        writer.write(mol)
        count += 1

writer.close()
print(f"✨ SUCCESS! {count} molecules saved to {output_file}")

In [ ]:
import os

new_folder = "LOTUS_SDF_Split"
os.makedirs(new_folder, exist_ok=True)

# Read the file we just created
suppl = Chem.SDMolSupplier("LOTUS_Final_Hits_2692.sdf")

print(f"🔪 Splitting into individual LOTUS ID files...")
count = 0

for mol in suppl:
    if mol is not None:
        lotus_id = mol.GetProp("_Name")
        file_path = os.path.join(new_folder, f"{lotus_id}.sdf")
        
        writer = Chem.SDWriter(file_path)
        writer.write(mol)
        writer.close()
        
        count += 1
        if count % 500 == 0:
            print(f"✅ Processed {count} files...")

print(f"✨ Done! {count} individual SDFs are in the '{new_folder}' folder.")
